# Redes Neuronales Recurrentes

Implementacion sobre una serie temporal sintetica mas compleja.

En los tres notebooks del bloque 2 vamos a usar las mismas funciones auxiliares, el mismo criterio de particion train/valid/test y el mismo esquema de logging en `Weights & Biases`.


## Librerias y configuracion general


In [ ]:
# Importamos librerías para generación de datos, entrenamiento, métricas y logging.

import random
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
from tqdm.auto import tqdm
import wandb


In [ ]:
# Fijamos semillas para reproducibilidad y detectamos el dispositivo disponible.

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


In [ ]:
# Configuramos el proyecto de WandB y pedimos la API key en forma interactiva.

from getpass import getpass

WANDB_PROJECT = "CEIA-Co[NUMERO]-RNN-signal"
WANDB_ENTITY = None

wandb_api_key = getpass('Pegue su API key de WandB: ')
login_ok = wandb.login(key=wandb_api_key, relogin=True, verify=True)
print('Login correcto en WandB:', login_ok)


## Funciones comunes


In [ ]:
# Generamos la serie temporal sintética usada por todos los experimentos del bloque.

def generate_time_series_complex(batch_size, n_steps, n_future=1, include_derivative=False):
    total_steps = n_steps + n_future
    time = np.linspace(0.0, 1.0, total_steps, dtype=np.float32)

    # generamos 3 frecuencias, fases y una envolvente de amplitud que varía lentamente, para cada serie del batch
    freq1 = np.random.uniform(8.0, 14.0, size=(batch_size, 1))
    freq2 = np.random.uniform(16.0, 26.0, size=(batch_size, 1))
    freq3 = np.random.uniform(2.0, 5.0, size=(batch_size, 1))

    phase1 = np.random.uniform(0.0, 2.0 * np.pi, size=(batch_size, 1))
    phase2 = np.random.uniform(0.0, 2.0 * np.pi, size=(batch_size, 1))
    phase3 = np.random.uniform(0.0, 2.0 * np.pi, size=(batch_size, 1))

    amplitude = 0.7 + 0.3 * np.sin(2.0 * np.pi * (time[None, :] * np.random.uniform(0.8, 1.5, size=(batch_size, 1)) + np.random.uniform(0.0, 1.0, size=(batch_size, 1))))

    # se contruyen las 3 series trigonometricas
    base = amplitude * np.sin(freq1 * time[None, :] + phase1)
    harmonic = 0.35 * np.sin(freq2 * time[None, :] + phase2)
    slow_component = 0.2 * np.cos(freq3 * time[None, :] + phase3)

    # se contruye ahora una tendencia lineal y una curvatura cuadrática, con coeficientes aleatorios para cada serie del batch
    slope = np.random.uniform(-0.5, 0.5, size=(batch_size, 1))
    trend = 0.25 * slope * (time[None, :] - 0.5)

    curvature = np.random.uniform(-0.15, 0.15, size=(batch_size, 1))
    quadratic = curvature * (time[None, :] - 0.5) ** 2

    # generamos un ruido
    noise = 0.08 * np.random.randn(batch_size, total_steps)

    # componemos la serie como suma de todo lo anterior
    series = base + harmonic + slow_component + trend + quadratic + noise

    # si se solicita, se calcula la derivada discreta de la serie y se concatena como una segunda feature
    if include_derivative:
        derivative = np.diff(series, axis=1, prepend=series[:, :1])
        series = np.stack([series, derivative], axis=-1)
    else:
        series = series[..., np.newaxis]

    return series.astype(np.float32)


In [ ]:
# Esta función muestra ejemplos de secuencias, targets y predicciones.

def plot_series(series, y=None, y_pred=None, rows=3, cols=4, title=None):
    series_main = series[:, :, 0]
    if y is not None and y.ndim == 3:
        y = y[:, :, 0]
    if y_pred is not None and isinstance(y_pred, torch.Tensor):
        y_pred = y_pred.detach().cpu().numpy()
    if y_pred is not None and y_pred.ndim == 3:
        y_pred = y_pred[:, :, 0]

    total = min(rows * cols, len(series_main))
    fig, axes = plt.subplots(rows, cols, figsize=(18, 10), sharex=True, sharey=True)
    axes = np.array(axes).reshape(rows, cols)

    for idx in range(rows * cols):
        ax = axes[idx // cols, idx % cols]
        if idx >= total:
            ax.axis('off')
            continue

        x = series_main[idx]
        ax.plot(np.arange(len(x)), x, '.-', label='input')
        if y is not None:
            ax.plot(np.arange(len(x), len(x) + y.shape[1]), y[idx], 'bx', markersize=8, label='target')
        if y_pred is not None:
            ax.plot(np.arange(len(x), len(x) + y_pred.shape[1]), y_pred[idx], 'ro', markersize=5, label='pred')
        ax.grid(True)
        ax.axhline(0.0, linewidth=1, color='gray')

    if title is not None:
        fig.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


In [ ]:
# Definimos el Dataset que entrega pares (X, y) para entrenamiento y solo X para test.

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y=None, train=True):
        self.X = X
        self.y = y
        self.train = train

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        if self.train:
            y = torch.from_numpy(self.y[idx])
            return x, y
        return x


In [ ]:
# Armamos train, validation y test a partir de la misma función generadora.

def build_dataloaders(n_steps=50, n_future=1, include_derivative=False, n_series=10000, batch_size=64):
    series = generate_time_series_complex(
        batch_size=n_series,
        n_steps=n_steps,
        n_future=n_future,
        include_derivative=include_derivative,
    )

    if include_derivative:
        y = series[:, -n_future:, 0]
    else:
        y = series[:, -n_future:, :]

    X = series[:, :n_steps]

    X_train, y_train = X[:7000], y[:7000]
    X_valid, y_valid = X[7000:9000], y[7000:9000]
    X_test, y_test = X[9000:], y[9000:]

    datasets = {
        'train': TimeSeriesDataset(X_train, y_train, train=True),
        'eval': TimeSeriesDataset(X_valid, y_valid, train=True),
        'test': TimeSeriesDataset(X_test, y_test, train=False),
    }

    dataloaders = {
        'train': DataLoader(datasets['train'], batch_size=batch_size, shuffle=True),
        'eval': DataLoader(datasets['eval'], batch_size=batch_size, shuffle=False),
        'test': DataLoader(datasets['test'], batch_size=batch_size, shuffle=False),
    }

    return dataloaders, (X_train, y_train, X_valid, y_valid, X_test, y_test)


In [ ]:
# Contamos parámetros y mostramos sus shapes para comparar capacidad entre modelos.

def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def imp_param(model):
    print('-' * 84)
    print('PARAMETROS DEL MODELO')
    print('-' * 84)
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f'{name}: {tuple(param.shape)}')
    print('-' * 84)
    print('TOTAL DE PARAMETROS ENTRENABLES:', count_parameters(model))
    print('-' * 84)


In [ ]:
# Función de entenamiento con early stopping y logging en WandB.
# Entrenamos un modelo y registramos train/eval loss en WandB.

def train_model(
    model,
    dataloaders,
    optimizer,
    project_name,
    run_name,
    config,
    num_epochs=20,
    patience=5,
    device=device,
    entity=WANDB_ENTITY,
):
    wandb.finish()
    run = wandb.init(
        project=project_name,
        entity=entity,
        name=run_name,
        config=config,
        reinit="finish_previous",
    )

    criterion = torch.nn.MSELoss()
    model.to(device)
    best_state = None
    best_eval = np.inf
    epochs_without_improvement = 0
    history = {'train_loss': [], 'eval_loss': []}

    for epoch in range(num_epochs):
        model.train()
        train_losses = []
        bar = tqdm(dataloaders['train'], leave=False)
        for X, y in bar:
            X = X.to(device)
            y = y.to(device).view(y.shape[0], -1)

            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            bar.set_description(f'train {np.mean(train_losses):.5f}')

        model.eval()
        eval_losses = []
        with torch.no_grad():
            for X, y in dataloaders['eval']:
                X = X.to(device)
                y = y.to(device).view(y.shape[0], -1)
                pred = model(X)
                loss = criterion(pred, y)
                eval_losses.append(loss.item())

        train_loss = float(np.mean(train_losses))
        eval_loss = float(np.mean(eval_losses))
        history['train_loss'].append(train_loss)
        history['eval_loss'].append(eval_loss)

        wandb.log({
            'epoch': epoch,
            'train_loss': train_loss,
            'eval_loss': eval_loss,
            'learning_rate': optimizer.param_groups[0]['lr'],
        })

        if eval_loss < best_eval:
            best_eval = eval_loss
            epochs_without_improvement = 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'Early stopping en epoch {epoch + 1}')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    wandb.summary['best_eval_loss'] = best_eval
    wandb.summary['num_param'] = count_parameters(model)
    wandb.finish()
    return history


In [ ]:
# Esta función genera predicciones sobre el conjunto de test y permite recortar contexto (muestras usadas para la predicción)

def predict_model(model, dataloader, reduced=0, device=device):
    model.eval()
    preds = []
    first_batch = True
    with torch.no_grad():
        for X in dataloader:
            if reduced and reduced > 0:
                X = X[:, -reduced:, :]
            if first_batch:
                print('X shape empleado para predecir:')
                print(tuple(X.shape))
                first_batch = False
            X = X.to(device)
            pred = model(X)
            preds.append(pred.detach().cpu())
    return torch.cat(preds, dim=0)


In [ ]:
# Calculamos el MSE final a partir de las predicciones del modelo.

def mse_from_predictions(y_true, y_pred):
    y_true = np.squeeze(y_true)
    y_pred = np.squeeze(y_pred.detach().cpu().numpy())
    return mean_squared_error(y_true, y_pred)


In [ ]:
# Baseline no recurrente: un MLP de dos capas que ve toda la ventana pasada.
# ATENTO!! el input_size se refiere a la cantidad de features por paso de tiempo
# (1 si no se incluye la derivada, 2 si se incluye),
# mientras que n_steps es la cantidad de pasos de tiempo que se ven como contexto. 
# El número total de entradas al MLP será n_steps * input_size.

class MLPRegressor(torch.nn.Module):
    def __init__(self, n_steps, input_size=1, hidden_dim=32, n_out=1):
        super().__init__()
        self.n_steps = n_steps
        self.input_size = input_size
        self.hidden_dim = hidden_dim
        self.n_out = n_out
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_steps * input_size, hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Linear(hidden_dim, n_out),
        )

    def forward(self, x):
        x = x.view(x.shape[0], -1)
        return self.net(x)

    # un metodo para usar en el config de WandB
    def model_config(self):
        return {
            'model_type': 'MLP',
            'n_steps': self.n_steps,
            'input_size': self.input_size,
            'hidden_dim': self.hidden_dim,
            'n_out': self.n_out,
            'num_param': count_parameters(self),
        }


In [ ]:
# Modelo recurrente base: procesa la secuencia y usa el último estado temporal.

class RNNRegressor(torch.nn.Module):
    def __init__(self, input_size=1, hidden_size=12, num_layers=1, n_out=1, nonlinearity='tanh'):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.n_out = n_out
        self.nonlinearity = nonlinearity
        self.rnn = torch.nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            nonlinearity=nonlinearity,
            batch_first=True,
        )
        self.fc = torch.nn.Linear(hidden_size, n_out)

    def forward(self, x):
        output, h_n = self.rnn(x)
        ultimo_estado = output[:, -1, :]
        return self.fc(ultimo_estado)

    def model_config(self):
        return {
            'model_type': 'RNN',
            'input_size': self.input_size,
            'hidden_size': self.hidden_size,
            'num_layers': self.num_layers,
            'n_out': self.n_out,
            'nonlinearity': self.nonlinearity,
            'num_param': count_parameters(self),
        }


## 1. Generacion de datos y visualizacion

En este bloque usamos una serie temporal sintetica mas compleja que la suma de dos senoidales simple.

La senal combina:
- varias frecuencias,
- modulacion lenta de amplitud,
- tendencia suave,
- curvatura,
- ruido.

Seguimos trabajando con `n_future = 1`.


In [ ]:
# Generamos el dataset del problema con la configuraci?n base del notebook.

N_STEPS = 50
N_FUTURE = 1
BATCH_SIZE = 64

dataloaders, data_splits = build_dataloaders(
    n_steps=N_STEPS,
    n_future=N_FUTURE,
    include_derivative=False,
    n_series=10000,
    batch_size=BATCH_SIZE,
)

X_train, y_train, X_valid, y_valid, X_test, y_test = data_splits

print('X_train, y_train:', X_train.shape, y_train.shape)
print('X_valid, y_valid:', X_valid.shape, y_valid.shape)
print('X_test, y_test:', X_test.shape, y_test.shape)


In [ ]:
# Visualizamos algunas series del conjunto de test.

plot_series(X_test, y=y_test, title='Serie temporal compleja - muestras de test')


## 2. MLP baseline

Usamos un `MLP` de dos capas como baseline. Es un modelo no recurrente: ve toda la ventana pasada pero no tiene un estado oculto temporal expl?cito.


In [ ]:
# Instanciamos el MLP baseline y revisamos su cantidad de parámetros.

mlp = MLPRegressor(n_steps=N_STEPS, input_size=1, hidden_dim=32, n_out=N_FUTURE)
print(mlp)
print(count_parameters(mlp))


In [ ]:
# Entrenamos el MLP baseline y guardamos sus métricas en WandB.

optimizer_mlp = torch.optim.Adam(mlp.parameters(), lr=1e-3)
mlp_history = train_model(
    model=mlp,
    dataloaders=dataloaders,
    optimizer=optimizer_mlp,
    project_name=WANDB_PROJECT,
    run_name='mlp-baseline',
    config=mlp.model_config(),
    num_epochs=20,
    patience=5,
)


In [ ]:
# Evaluamos el MLP en test y graficamos sus predicciones.

mlp.to(device)
y_pred_mlp = predict_model(mlp, dataloaders['test'])
mlp_mse = mse_from_predictions(y_test, y_pred_mlp)
print('MSE MLP baseline:', mlp_mse)
plot_series(X_test, y=y_test, y_pred=y_pred_mlp, title='MLP baseline - prediccion a 1 paso')


## 3. RNN base

Ahora implementamos una `RNN` basica con una capacidad algo mayor que el ejemplo minimo del bloque anterior, pero todavia simple de entender e inspeccionar.


In [ ]:
# Instanciamos la RNN base del docente y revisamos su capacidad.

rnn_base = RNNRegressor(input_size=1, hidden_size=12, num_layers=1, n_out=N_FUTURE, nonlinearity='tanh')
print(rnn_base)
imp_param(rnn_base)


In [ ]:
# Verificamos las dimensiones de entrada y salida de la RNN con un forward simple.

entrada = torch.rand(4, N_STEPS, 1)
salida = rnn_base(entrada)
print('entrada shape:', tuple(entrada.shape))
print('salida shape:', tuple(salida.shape))


In [ ]:
# Entrenamos la RNN base sobre la misma serie temporal.

optimizer_rnn = torch.optim.Adam(rnn_base.parameters(), lr=1e-3)
rnn_history = train_model(
    model=rnn_base,
    dataloaders=dataloaders,
    optimizer=optimizer_rnn,
    project_name=WANDB_PROJECT,
    run_name='rnn-base',
    config=rnn_base.model_config(),
    num_epochs=20,
    patience=5,
)


In [ ]:
# Evaluamos la RNN base y graficamos sus predicciones.

rnn_base.to(device)
y_pred_rnn = predict_model(rnn_base, dataloaders['test'])
rnn_mse = mse_from_predictions(y_test, y_pred_rnn)
print('MSE RNN base:', rnn_mse)
plot_series(X_test, y=y_test, y_pred=y_pred_rnn, title='RNN base - prediccion a 1 paso')


## 4. Analisis de memoria temporal de la RNN

Medimos como se degrada el error cuando la RNN entrenada recibe menos historia pasada.

El argumento `reduced` conserva solamente las ultimas `k` muestras de la ventana de entrada.


In [ ]:
# Medimos cómo cambia el error cuando la RNN recibe menos historia pasada.

errores_reduced = []
for k in range(1, N_STEPS + 1):
    y_pred_reduced = predict_model(rnn_base, dataloaders['test'], reduced=k)
    error_k = mse_from_predictions(y_test, y_pred_reduced)
    errores_reduced.append(error_k)

plt.figure(figsize=(8, 4))
plt.plot(range(1, N_STEPS + 1), errores_reduced)
plt.xlabel('Numero de muestras pasadas conservadas')
plt.ylabel('MSE')
plt.grid(True)
plt.show()


## 5. Comparacion entre modelos

Comparamos el error final y la cantidad de par?metros entre el `MLP` baseline y la `RNN` base.


In [ ]:
# Resumimos error final y número de parámetros para comparar ambos modelos.

resumen = [
    {
        'modelo': 'MLP baseline',
        'parámetros': count_parameters(mlp),
        'mse_test': mlp_mse,
    },
    {
        'modelo': 'RNN base',
        'parámetros': count_parameters(rnn_base),
        'mse_test': rnn_mse,
    },
]

for fila in resumen:
    print(fila)
